# Camada Gold: Resumo Diário de Vendas
**Módulo: Agregações Executivas e KPIs (Visão de Negócio)**

> Esta tabela (`gold.resumo_vendas_diarias`) consolida os indicadores financeiros e operacionais do e-commerce. Diferente das camadas anteriores, ela atua como uma "Vitrine Blindada", otimizada e modelada dimensionalmente para consumo direto em ferramentas de BI (como Power BI) e tomada de decisão pela diretoria.

**Granularidade:** Agrupamento diário por Status do Pedido e Método de Pagamento.

**KPIs e Agregações Geradas por este Arquivo:**

* **KPI 1 - Volume de Vendas:** Contagem total de pedidos realizados (`qtd_pedidos`).
* **KPI 2 - Faturamento Total:** Soma financeira da coluna valor total (`faturamento_total`).
* **KPI 3 - Ticket Médio:** Cálculo do valor médio gasto por pedido (`ticket_medio`).
* **KPI 4 - Custo de Frete:** Soma dos valores de frete para análise logística (`frete_total`).

# 1. Carga do Motor Utilitário
**Isolamento do Comando Mágico**

> Devido a particularidades do interpretador do Databricks, o comando `%run` deve ser executado em uma célula isolada, sem comentários ou variáveis, garantindo que as funções globais da Squad (como conexões e logs) sejam injetadas no ambiente corretamente.

In [0]:
%run ../99_utils/feat_squad2_99_helpers

# 2. Configuração do Ambiente Gold
**Importação de Bibliotecas e Definição de Variáveis**

> Carregamento das funções de agregação matemática do PySpark (`sum`, `count`, `round`) e do formato Delta para a operação de UPSERT (Merge). Definição das tabelas de origem (Silver) e destino (Gold) para parametrização do pipeline.

In [0]:
# 1. Importações específicas para a Gold (Matemática e Agregações)
from pyspark.sql.functions import col, sum as _sum, count, round, date_format, current_timestamp, lit
from deltalake import DeltaTable

# 2. Definição das tabelas de ponte
TABELA_ORIGEM = "ecommerce_pedidos"
TABELA_DESTINO = "resumo_vendas_diarias"

log.info("🏆 Ambiente Gold preparado e bibliotecas carregadas com sucesso!")

# 3. Motor de Agregações (KPIs de Negócio)
**Transformação de Dados Brutos em Indicadores Gerenciais**

> Aqui aplica-se a regra de negócio para a construção do painel executivo:
* **Dimensões (Agrupamento):** Data da Venda (extraída do timestamp original), Status do Pedido e Método de Pagamento.
* **Métricas (Fatos):** Quantidade de Pedidos (Volume Operacional), Faturamento Total, Custo de Frete Total e Ticket Médio (Faturamento / Quantidade).

In [0]:
try:
    log.info("📊 Iniciando o motor de agregações da Camada Gold...")
    
    # 1. Lê os dados higienizados da Camada Silver
    df_silver = ler_delta("silver", TABELA_ORIGEM)
    
    # 2. Aplica a modelagem dimensional e os cálculos matemáticos
    df_gold_agg = df_silver \
        .withColumn("dt_venda", date_format(col("dt_pedido"), "yyyy-MM-dd")) \
        .groupBy("dt_venda", "status_pedido", "metodo_pagamento") \
        .agg(
            count("id_pedido").alias("qtd_pedidos"),
            round(_sum("valor_total"), 2).alias("faturamento_total"),
            round(_sum("valor_frete"), 2).alias("custo_frete_total")
        ) \
        .withColumn("ticket_medio", round(col("faturamento_total") / col("qtd_pedidos"), 2))
        
    log.info(f"✅ Agregação concluída com sucesso! Gerando prévia dos indicadores:")
    display(df_gold_agg.orderBy("dt_venda", "status_pedido").limit(10))

except Exception as e:
    log.error(f"❌ Erro na etapa de agregação: {str(e)}")
    raise

# 4. Persistência e Serving Layer (Dupla Gravação)
**Distribuição dos Indicadores para o Data Lake e SQL Server**

> Para atender aos requisitos de arquitetura da Squad e garantir performance no BI (Looker):
1. **Data Lake (Fonte da Verdade):** Os dados agregados são salvos em formato Delta na camada Gold.
2. **SQL Server (Serving Layer):** Uma cópia idêntica é enviada via conexão JDBC para o banco de dados relacional, otimizando o tempo de resposta das consultas do painel executivo.

In [0]:
try:
    log.info("💾 Iniciando a Dupla Gravação da Camada Gold...")
    
    # ==============================================================================
    # 1. GRAVAÇÃO NO DATA LAKE (DELTA)
    # ==============================================================================
    log.info("🌊 Salvando no Azure Data Lake (Formato Delta)...")
    sucesso_lake = gravar_delta(df_gold_agg, "gold", TABELA_DESTINO, mode="overwrite")
    
    # ==============================================================================
    # 2. GRAVAÇÃO NO SQL SERVER (SERVING LAYER)
    # ==============================================================================
    if sucesso_lake:
        log.info("🔌 Conectando ao SQL Server da Serving Layer...")
        
        # Resgata as variáveis de ambiente carregadas pelo helpers
        sql_host = os.getenv("SQL_HOST")
        sql_db = os.getenv("SQL_DATABASE")
        sql_user = os.getenv("SQL_USERNAME")
        sql_pass = os.getenv("SQL_PASSWORD")
        
        # Define o nome da tabela no banco de dados (usando o prefixo da squad)
        tabela_sql = f"{SQL_SCHEMA}_{TABELA_DESTINO}"
        
        log.info(f"🚀 Enviando dados para a tabela [{tabela_sql}] no banco [{sql_db}] (Modo Serverless)...")
        
        # Executa o push quebrando a URL em parâmetros isolados (Exigência do Serverless)
        df_gold_agg.write \
            .format("sqlserver") \
            .option("host", sql_host) \
            .option("port", "1433") \
            .option("database", sql_db) \
            .option("dbtable", tabela_sql) \
            .option("user", sql_user) \
            .option("password", sql_pass) \
            .option("encrypt", "true") \
            .option("trustServerCertificate", "false") \
            .mode("overwrite") \
            .save()
        
        log.info("✅ SUCESSO ABSOLUTO! Pipeline Gold finalizado. Dados disponíveis para o Looker!")

except Exception as e:
    log.error(f"❌ Erro na gravação da Camada Gold: {str(e)}")
    raise

## 5. Insights Gráficos de Negócio (Overdelivery Executivo)

**Painel Gerencial e Visão Estratégica**
> Para agregar valor imediato à entrega da Squad, esta etapa gera visualizações executivas baseadas na nossa *Serving Layer*. O objetivo é permitir que as áreas de Negócio identifiquem rapidamente padrões de consumo, saúde da operação e a sazonalidade de faturamento ao longo do tempo, sem precisarem esperar a construção do dashboard final na ferramenta de BI.

**Linhagem e Motor de Transformação (Visão Técnica)**
> Para fins de observabilidade e manutenção do time de Engenharia, os gráficos abaixo são processados dinamicamente em memória a partir do DataFrame final `df_gold_agg`. As transformações aplicadas são:

* **Gráfico 1 (Financeiro):** Agrupamento do dataframe pela dimensão `metodo_pagamento` e aplicação da função de agregação `sum()` sobre a métrica `faturamento_total`.
* **Gráfico 2 (Operacional):** Agrupamento pela dimensão `status_pedido` e soma matemática (`sum()`) da volumetria exata de `qtd_pedidos`.
* **Gráfico 3 (Sazonalidade):** Extração do padrão `yyyy-MM` da coluna de data (`dt_venda`) via função nativa `date_format`, seguida de agrupamento e soma do `faturamento_total` para projetar a linha do tempo cronológica.

In [0]:
# ==============================================================================
# 5. Insights Gráficos de Negócio (Overdelivery Executivo)
# ==============================================================================
import pyspark.sql.functions as F

try:
    log.info("📊 Gerando painéis executivos de negócio a partir da Camada Gold...")
    
    # Insight 1: Faturamento por Método de Pagamento
    print("\n💰 INSIGHT FINANCEIRO: Faturamento Total por Método de Pagamento")
    df_metodo_pagamento = df_gold_agg.groupBy("metodo_pagamento") \
                                     .agg(F.sum("faturamento_total").alias("Faturamento_Total")) \
                                     .orderBy(F.desc("Faturamento_Total"))
    
    display(df_metodo_pagamento)


    # Insight 2: Saúde da Operação (Pedidos por Status)
    print("\n📦 INSIGHT OPERACIONAL: Volume de Pedidos por Status Atual")
    df_status = df_gold_agg.groupBy("status_pedido") \
                           .agg(F.sum("qtd_pedidos").alias("Total_de_Pedidos")) \
                           .orderBy(F.desc("Total_de_Pedidos"))
    
    display(df_status)


    # =======================================================================
    # Insight 3: Sazonalidade de Vendas (100% Dinâmico - Sem Hardcoding)
    # =======================================================================
    print("\n📈 INSIGHT TEMPORAL: Tendência de Faturamento por Mês/Ano")
    
    # 1. Força o início no primeiro mês do ano (Jan) e o fim no último mês do ano (Dez)
    df_limites = df_gold_agg.select(
        F.date_trunc("year", F.min("dt_venda")).alias("inicio_ano"),
        F.add_months(F.date_trunc("year", F.max("dt_venda")), 11).alias("fim_ano")
    )

    # 2. Gera a sequência completa de 12 meses para o ano correspondente
    df_calendario = df_limites.select(
        F.explode(
            F.sequence(
                F.col("inicio_ano"),
                F.col("fim_ano"),
                F.expr("interval 1 month")
            )
        ).alias("data_sequencial")
    ).withColumn("mes_ref", F.date_format("data_sequencial", "yyyy-MM")).select("mes_ref")

    # 3. Agrupa os dados reais de vendas que temos na Gold
    df_vendas_reais = df_gold_agg.withColumn("mes_venda", F.date_format("dt_venda", "yyyy-MM")) \
                                 .groupBy("mes_venda") \
                                 .agg(F.sum("faturamento_total").alias("Faturamento_Total"))
    
    # 4. Cruza o esqueleto anual com as vendas reais (Left Join)
    df_temporal = df_calendario.join(df_vendas_reais, df_calendario.mes_ref == df_vendas_reais.mes_venda, "left") \
                               .select(F.col("mes_ref").alias("mes_ano"), F.col("Faturamento_Total"))
    
    # 5. Preenche os meses sem movimentação com 0.0 para manter a linha contínua
    df_temporal_final = df_temporal.fillna(0, subset=["Faturamento_Total"]).orderBy("mes_ano")
    
    # Exibe o resultado pronto para o gráfico de linhas
    display(df_temporal_final)

except Exception as e:
    log.error(f"❌ Erro ao gerar os insights de negócio: {str(e)}")

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.